
# 4) Alignment and Expansion

A private knowledge graph is useful, but it stays small if it only relies on our extracted data. Alignment connects our entities to a larger open knowledge base, which gives us two advantages:

- we avoid duplicating entities that already exist in Wikidata;
- we can expand the graph in a principled way from trusted external identifiers.

This step is critical because poor alignment creates poor expansion. If we align the wrong entity, the KB may grow quickly but become semantically incorrect. Therefore, we use a confidence score and keep a manual-review category instead of aligning everything automatically.



## Alignment Workflow
For each important private entity, we follow this workflow:

1. query candidate matches from Wikidata;
2. compare labels;
3. compare the expected type or description;
4. compute a confidence score;
5. decide whether to:
   - accept the alignment with `owl:sameAs`;
   - flag the case for manual review;
   - keep the entity local if no safe alignment is found.

This notebook focuses on **important entities** rather than every node. We prioritize:
- all tournaments;
- all surfaces;
- all country-like nodes;
- the most frequent players in the current graph.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from rdflib import Graph

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.kg.alignment_and_expansion import (
    HEADERS,
    WIKIDATA_SPARQL_URL,
    build_alignment_and_expansion,
    run_select_query,
)
from src.kg.kg_construction import build_kg_artifacts

TRIPLES_PATH = PROJECT_ROOT / "data" / "interim" / "extracted_knowledge.csv"
INITIAL_KG_PATH = PROJECT_ROOT / "kg_artifacts" / "initial_kg.ttl"
ALIGNMENT_TABLE_PATH = PROJECT_ROOT / "kg_artifacts" / "alignment_table.csv"
ALIGNMENT_TTL_PATH = PROJECT_ROOT / "kg_artifacts" / "alignment.ttl"
PREDICATE_ALIGNMENT_TTL_PATH = PROJECT_ROOT / "kg_artifacts" / "predicate_alignment.ttl"
EXPANDED_TTL_PATH = PROJECT_ROOT / "kg_artifacts" / "expanded_kg.ttl"
EXPANDED_NT_PATH = PROJECT_ROOT / "kg_artifacts" / "expanded_kg.nt"
EXPANDED_STATS_PATH = PROJECT_ROOT / "kg_artifacts" / "expanded_stats.json"

print(f"Input candidate triples: {TRIPLES_PATH}")
print(f"Initial KG: {INITIAL_KG_PATH}")

Input candidate triples: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_knowledge.csv
Initial KG: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/initial_kg.ttl



## Load the Private Graph and the Cleaned Triple Table
We start from the initial RDF graph created in Notebook 3 and the cleaned triple table that represents our private tennis KB.


In [ ]:
triples_df = pd.read_csv(TRIPLES_PATH)
initial_kg = Graph()
initial_kg.parse(INITIAL_KG_PATH, format="turtle")

kg_artifacts = build_kg_artifacts(triples_df)
print(f"Initial KG triples: {len(initial_kg)}")
print(f"Cleaned private triples available for alignment support: {len(kg_artifacts.cleaned_triples)}")

Initial KG triples: 1787


Cleaned private triples available for alignment support: 640



## Confidence Rule
We use the following decision rule in this notebook:
- `score >= 0.90` -> accept alignment automatically;
- `0.70 <= score < 0.90` -> manual review;
- `score < 0.70` -> do not align automatically and keep a local entity.

This is a conservative strategy. In academic work, it is usually better to review uncertain matches than to inject wrong `sameAs` links.



## Build the Alignment Table and the Expanded Graph
The next cell performs the full workflow:
- select important entities;
- search Wikidata candidates;
- compute confidence scores;
- generate the alignment table;
- export `owl:sameAs` links;
- expand the graph from confidently aligned seeds.

This may take some time because it performs real external KB queries.


In [ ]:
alignment_artifacts = build_alignment_and_expansion(triples_df, initial_kg)

alignment_artifacts.alignment_df.to_csv(ALIGNMENT_TABLE_PATH, index=False)
alignment_artifacts.alignment_graph.serialize(ALIGNMENT_TTL_PATH, format="turtle")
alignment_artifacts.predicate_alignment_graph.serialize(PREDICATE_ALIGNMENT_TTL_PATH, format="turtle")
alignment_artifacts.expanded_graph.serialize(EXPANDED_TTL_PATH, format="turtle")
alignment_artifacts.expanded_graph.serialize(EXPANDED_NT_PATH, format="nt")
with EXPANDED_STATS_PATH.open("w", encoding="utf-8") as file:
    json.dump(alignment_artifacts.expanded_stats, file, indent=2)

print(f"Saved alignment table to: {ALIGNMENT_TABLE_PATH}")
print(f"Saved alignment graph to: {ALIGNMENT_TTL_PATH}")
print(f"Saved predicate alignment graph to: {PREDICATE_ALIGNMENT_TTL_PATH}")
print(f"Saved expanded graph to: {EXPANDED_TTL_PATH}")

/opt/anaconda3/lib/python3.13/site-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Saved alignment table to: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/alignment_table.csv
Saved alignment graph to: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/alignment.ttl
Saved predicate alignment graph to: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/predicate_alignment.ttl
Saved expanded graph to: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/expanded_kg.ttl



## Mandatory Alignment Table
This table is the central deliverable of the alignment step. It records:
- the private entity;
- the external URI candidate;
- the confidence score;
- the final decision.


In [4]:
alignment_df = alignment_artifacts.alignment_df.copy()
display(alignment_df.head(25))

,private_entity,entity_role,external_uri,candidate_label,candidate_description,confidence,decision
0,New Zealand,Country,http://www.wikidata.org/entity/Q664,New Zealand,island country in the southwest Pacific Ocean,1.00,accept_alignment
1,Jamaica,Country,http://www.wikidata.org/entity/Q766,Jamaica,island state in the Caribbean Sea,1.00,accept_alignment
2,Spain,Country,http://www.wikidata.org/entity/Q29,Spain,manual high-confidence seed,0.99,accept_alignment
3,Australia,Country,http://www.wikidata.org/entity/Q408,Australia,manual high-confidence seed,0.99,accept_alignment
4,David Ferrer,Player,http://www.wikidata.org/entity/Q191476,David Ferrer,Spanish tennis player (born 1982),1.00,accept_alignment
5,Dominic Thiem,Player,http://www.wikidata.org/entity/Q88762,Dominic Thiem,Austrian male tennis player,1.00,accept_alignment
6,Gaël Monfils,Player,http://www.wikidata.org/entity/Q186429,Gaël Monfils,French tennis player,1.00,accept_alignment
7,Kei Nishikori,Player,http://www.wikidata.org/entity/Q311222,Kei Nishikori,Japanese tennis player,1.00,accept_alignment
8,John Isner,Player,http://www.wikidata.org/entity/Q53566,John Isner,American tennis player,1.00,accept_alignment
9,Casper Ruud,Player,http://www.wikidata.org/entity/Q18810082,Casper Ruud,Norwegian tennis player,1.00,accept_alignment



## Alignment Summary
We now summarize how many entities were accepted, sent to manual review, or kept local.


In [5]:
alignment_summary = alignment_df['decision'].value_counts().rename_axis('decision').reset_index(name='count')
display(alignment_summary)

,decision,count
0,accept_alignment,42
1,manual_review,14
2,create_local_entity,4



## Accepted, Review, and Local Examples
A good notebook does not only show the accepted links. It should also explain why some cases are uncertain.

Typical accepted cases in this project are well-known players and the four Grand Slam tournaments. Typical manual-review cases are ambiguous place names mistakenly extracted as countries. Local-only cases appear when the label is too partial or too ambiguous.


In [ ]:
accepted_examples = alignment_df[alignment_df['decision'] == 'accept_alignment'].head(12)
review_examples = alignment_df[alignment_df['decision'] == 'manual_review'].head(12)
local_examples = alignment_df[alignment_df['decision'] == 'create_local_entity'].head(12)

print('Accepted examples:')
display(accepted_examples)
print('Manual-review examples:')
display(review_examples)
print('Local entities kept without automatic alignment:')
display(local_examples)

Accepted examples:


,private_entity,entity_role,external_uri,candidate_label,candidate_description,confidence,decision
0,New Zealand,Country,http://www.wikidata.org/entity/Q664,New Zealand,island country in the southwest Pacific Ocean,1.00,accept_alignment
1,Jamaica,Country,http://www.wikidata.org/entity/Q766,Jamaica,island state in the Caribbean Sea,1.00,accept_alignment
2,Spain,Country,http://www.wikidata.org/entity/Q29,Spain,manual high-confidence seed,0.99,accept_alignment
3,Australia,Country,http://www.wikidata.org/entity/Q408,Australia,manual high-confidence seed,0.99,accept_alignment
4,David Ferrer,Player,http://www.wikidata.org/entity/Q191476,David Ferrer,Spanish tennis player (born 1982),1.00,accept_alignment
5,Dominic Thiem,Player,http://www.wikidata.org/entity/Q88762,Dominic Thiem,Austrian male tennis player,1.00,accept_alignment
6,Gaël Monfils,Player,http://www.wikidata.org/entity/Q186429,Gaël Monfils,French tennis player,1.00,accept_alignment
7,Kei Nishikori,Player,http://www.wikidata.org/entity/Q311222,Kei Nishikori,Japanese tennis player,1.00,accept_alignment
8,John Isner,Player,http://www.wikidata.org/entity/Q53566,John Isner,American tennis player,1.00,accept_alignment
9,Casper Ruud,Player,http://www.wikidata.org/entity/Q18810082,Casper Ruud,Norwegian tennis player,1.00,accept_alignment


Manual-review examples:


,private_entity,entity_role,external_uri,candidate_label,candidate_description,confidence,decision
46,Miami,Country,http://www.wikidata.org/entity/Q8652,Miami,"city in Miami-Dade County, Florida, United States",0.75,manual_review
47,Jerez,Country,http://www.wikidata.org/entity/Q37114683,Jerez,family name,0.75,manual_review
48,Victoria,Country,http://www.wikidata.org/entity/Q590893,Victoria,female given name,0.75,manual_review
49,Munich,Country,http://www.wikidata.org/entity/Q1726,Munich,"capital and most populous city of Bavaria, Ger...",0.75,manual_review
50,London,Country,http://www.wikidata.org/entity/Q84,London,capital and largest city of England and the Un...,0.75,manual_review
51,Madrid,Country,http://www.wikidata.org/entity/Q2807,Madrid,capital of Spain,0.75,manual_review
52,Majorca,Country,http://www.wikidata.org/entity/Q105954479,Majorca,ship built in 1957,0.75,manual_review
53,Manacor,Country,http://www.wikidata.org/entity/Q49567,Manacor,"town and municipality in Mallorca, Spain",0.75,manual_review
54,Paris,Country,http://www.wikidata.org/entity/Q90,Paris,capital and most populous city in France,0.75,manual_review
55,Tomas Berdych,Player,http://www.wikidata.org/entity/Q193665,Tomáš Berdych,Czech tennis player,0.80,manual_review


Local entities kept without automatic alignment:


,private_entity,entity_role,external_uri,candidate_label,candidate_description,confidence,decision
42,Del Potro,Player,http://www.wikidata.org/entity/Q180535,Juan Martín del Potro,Argentine tennis player,0.640,create_local_entity
43,del Potro,Player,http://www.wikidata.org/entity/Q180535,Juan Martín del Potro,Argentine tennis player,0.640,create_local_entity
44,Grass,Surface,http://www.wikidata.org/entity/Q1151291,grass court,type of tennis court,0.656,create_local_entity
45,Clay,Surface,http://www.wikidata.org/entity/Q198718,clay court,type of tennis court,0.621,create_local_entity



## New Entity Creation Policy
When an entity is not found safely in the external KB, we do **not** force an alignment. Instead, we keep it local and make sure it still has:
- a local URI;
- a class in the ontology;
- a place inside the private graph.

This rule is important because a wrong external link is often worse than a local entity.



## Predicate Alignment Discussion
Entity alignment is only part of the work. We also need to study how our local predicates relate to Wikidata properties.

Two useful Semantic Web notions are:
- `owl:equivalentProperty`: two properties are semantically equivalent;
- `rdfs:subPropertyOf`: one property is more specific than another.

In practice, some of our predicates map cleanly to Wikidata, while others are only approximate:
- `fromCountry` is close to Wikidata `country of citizenship`;
- `hasSurface` is close to `surface played on`;
- `won` and `participatedIn` often require more context, because Wikidata may model wins at the tournament-edition level instead of directly between player and tournament.



## SPARQL Predicate Alignment Example 1
The query below follows the lab idea of checking which property directly links two aligned entities. Here we use:
- Rafael Nadal -> Wikidata `Q10132`
- Spain -> Wikidata `Q29`

This helps us study which external property could support our local predicate `fromCountry`.


In [7]:
predicate_query_country = """
SELECT ?property WHERE {
  wd:Q10132 ?property wd:Q29 .
}
"""

country_property_rows = run_select_query(predicate_query_country)
pd.DataFrame(country_property_rows).head()

,property
0,"{'type': 'uri', 'value': 'http://www.wikidata...."
1,"{'type': 'uri', 'value': 'http://www.wikidata...."



In this case, one of the returned properties is `P27`, which corresponds to **country of citizenship**. This is a strong candidate mapping for our local predicate `fromCountry`.



## SPARQL Predicate Alignment Example 2
The next query is a text-based candidate search, as required by the lab instructions. We search Wikidata property labels containing the word `surface`.


In [ ]:
predicate_text_search_query = """
SELECT ?property ?propertyLabel WHERE {
  ?property a wikibase:Property .
  ?property rdfs:label ?propertyLabel .
  FILTER(CONTAINS(LCASE(?propertyLabel), "surface"))
  FILTER(LANG(?propertyLabel) = "en")
}
LIMIT 20
"""

surface_property_rows = run_select_query(predicate_text_search_query)
surface_property_df = pd.DataFrame(
    [
        {
            'property': row['property']['value'],
            'propertyLabel': row['propertyLabel']['value'],
        }
        for row in surface_property_rows
    ]
)
display(surface_property_df.head(10))

,property,propertyLabel
0,http://www.wikidata.org/entity/P3013,surface tension
1,http://www.wikidata.org/entity/P765,surface played on
2,http://www.wikidata.org/entity/P2856,EU Surface Water Body Code
3,http://www.wikidata.org/entity/P7015,surface gravity
4,http://www.wikidata.org/entity/P7083,surface roughness
5,http://www.wikidata.org/entity/P10614,has surface



Among these candidates, `P765` (`surface played on`) is a plausible external match for our local predicate `hasSurface`. This still requires manual validation, because label similarity alone is not enough to guarantee semantic equivalence.



## How We Validate Predicate Mappings Manually
A candidate property is only accepted after checking:
- whether the linked subject and object types are compatible;
- whether the semantics are really the same and not just close;
- whether Wikidata models the fact at the same level of granularity.

For example, a player may *win* a **tournament edition** in Wikidata, while our current local graph may connect the player to the tournament more directly. In that case, `rdfs:subPropertyOf` may be more honest than `owl:equivalentProperty`.



## Expansion Strategy
We expand **only from confidently aligned entities**.

The strategy used here is:
1. **1-hop expansion** from accepted seeds;
2. **controlled 2-hop expansion** from a bounded sample of first-hop neighbors;
3. predicate filtering so the final graph remains manageable on a laptop.

This follows the lab recommendation that expansion should be anchored on reliable alignments and cleaned before embedding.



## Expanded Graph Statistics
We now inspect the size of the expanded KB after merging the local graph, the accepted `owl:sameAs` links, and the external Wikidata triples.


In [ ]:
with EXPANDED_STATS_PATH.open("r", encoding="utf-8") as file:
    expanded_stats = json.load(file)

print(json.dumps(expanded_stats, indent=2))

{
  "total_triples": 60443,
  "total_entities": 29830,
  "total_relations": 139
}



## Cleaning Before Embedding
Before exporting the expanded graph for embeddings, we already applied several cleaning steps:
- duplicate triples were removed automatically by graph semantics;
- only confidently aligned seeds were used for expansion;
- predicate variety was bounded to keep the KB manageable;
- local and external data were merged through explicit `owl:sameAs` links.

This does not make the graph perfect, but it creates a much stronger base for KGE than the small private graph alone.


In [10]:
expanded_graph = Graph()
expanded_graph.parse(EXPANDED_TTL_PATH, format="turtle")
print(f"Reloaded expanded graph triples: {len(expanded_graph)}")

sample_expanded_triples = list(expanded_graph)[:15]
for triple in sample_expanded_triples:
    print(triple)

Reloaded expanded graph triples: 60443
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q1033'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P2936'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q3441299'))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q736'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P463'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q123759'))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q869'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P706'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q11708'))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q8652'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P31'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q1093829'))
(rdflib.term.URIRef('http://www.wikidata.org/entity/Q736'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P2936'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q12953847'))
(rdflib.term.URIRef('http

This notebook produces the following artifacts:
- `kg_artifacts/alignment_table.csv`
- `kg_artifacts/alignment.ttl`
- `kg_artifacts/expanded_kg.ttl`
- `kg_artifacts/expanded_kg.nt`
- `kg_artifacts/expanded_stats.json`

These files will be the basis for Notebook 5, where we will perform SWRL reasoning and train knowledge graph embedding models.
